In [14]:
import yfinance as yf
import pandas as pd

# Load Dataset

In [68]:
tickers = {
    # Banking
    "BBRI.JK":  "Bank Rakyat Indonesia",
    "BMRI.JK":  "Bank Mandiri",
    # Telco & Infrastructure
    "TLKM.JK":  "Telkom Indonesia",
}

In [69]:
# Load dataset from the tickers
period = "3y"       # historical data window
print(f"\n📥 Downloading {period} of price data for {len(tickers)} assets...")
df = yf.download(list(tickers.keys()), period=period, auto_adjust=True, progress=False)["Close"]

# yfinance returns MultiIndex columns (Price, Ticker) even after ["Close"]
# Flatten to single level so rename() works correctly
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(-1)


📥 Downloading 3y of price data for 3 assets...


In [70]:
# Rename columns to friendly names
df.rename(columns=tickers, inplace=True)

# Forward-fill remaining gaps (weekends, holidays)
df = df.ffill().dropna()

dropped = set(tickers.values()) - set(df.columns)
if dropped:
    print(f"⚠️  Dropped due to insufficient data: {dropped}")

print(f"✅ Clean data: {df.shape[0]} trading days × {df.shape[1]} assets")
print(f"   Period: {df.index[0].date()} → {df.index[-1].date()}\n")

✅ Clean data: 709 trading days × 3 assets
   Period: 2023-03-16 → 2026-03-16



In [71]:
df

Ticker,Bank Rakyat Indonesia,Bank Mandiri,Telkom Indonesia
Date,,,
2023-03-16,3771.247314,3970.152588,3395.231201
2023-03-17,3906.789551,4070.917480,3395.231201
2023-03-20,3890.843018,4030.611084,3344.931641
2023-03-21,3906.789551,4232.142090,3453.914551
2023-03-24,3991.490723,4393.366699,3411.998047
...,...,...,...
2026-03-10,3560.000000,4910.000000,2960.000000
2026-03-11,3580.000000,4880.000000,3000.000000
2026-03-12,3570.000000,4960.000000,3020.000000


# Estimate Expected Returns & Covariance Matrix

After reading your historical prices into a pandas dataframe df, you need to decide between the available methods for estimating expected returns and the covariance matrix. Sensible defaults are `expected_returns.mean_historical_return()` and the Ledoit Wolf shrinkage estimate of the covariance matrix found in `risk_models.CovarianceShrinkage`. It is simply a matter of applying the relevant functions to the price dataset:

In [72]:
from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import CovarianceShrinkage

mu = mean_historical_return(df)
S = CovarianceShrinkage(df).ledoit_wolf()

`mu` will then be a pandas series of estimated expected returns for each asset, and `S` will be the estimated covariance matrix (part of it is shown below):

In [73]:
mu

Ticker
Bank Rakyat Indonesia   -0.028202
Bank Mandiri             0.061907
Telkom Indonesia        -0.047655
dtype: float64

In [74]:
S

Ticker,Bank Rakyat Indonesia,Bank Mandiri,Telkom Indonesia
Ticker,,,
Bank Rakyat Indonesia,0.089839,0.056893,0.026496
Bank Mandiri,0.056893,0.095651,0.030847
Telkom Indonesia,0.026496,0.030847,0.100985


# Mean-variance optimization
Mean-variance optimization is based on Harry Markowitz’s 1952 classic paper [1], which spearheaded the transformation of portfolio management from an art into a science. The key insight is that by combining assets with different expected returns and volatilities, one can decide on a mathematically optimal allocation.

If $w$ is the weight vector of stocks with expected returns $\mu$, then the portfolio return is equal to each stock’s weight multiplied by its return, i.e $w^T\mu$. The portfolio risk in terms of the covariance matrix $\Sigma$ is given by $w^T \Sigma w$. Portfolio optimization can then be regarded as a convex optimization problem, and a solution can be found using quadratic programming. If we denote the target return as $\mu^*$ the precise statement of the long-only portfolio optimization problem is as follows:

$$
\begin{aligned}
\min_{w} \quad & w^{T}\Sigma w \\
\text{subject to} \quad 
& w^{T}\mu \geq \mu^{*} \\
& w^{T}\mathbf{1} = 1 \\
& w_i \geq 0
\end{aligned}
$$

If we vary the target return, we will get a different set of weights (i.e a different portfolio) – the set of all these optimal portfolios is referred to as the efficient frontier.

![Optimization Problem](efficient_frontier.webp)

The Sharpe ratio is the portfolio’s return in excess of the risk-free rate, per unit risk (volatility).

$$
SR = \frac{R_p - R_f}{\sigma}
$$

It is particularly important because it measures the portfolio returns, adjusted for risk. So in practice, rather than trying to minimise volatility for a given target return (as per Markowitz 1952), it often makes more sense to just find the portfolio that maximises the Sharpe ratio. This is implemented as the `max_sharpe()` method in the `EfficientFrontier` class. Using the series `mu` and dataframe `S` from before:

In [75]:
from pypfopt.efficient_frontier import EfficientFrontier

ef = EfficientFrontier(mu, S)
weights = ef.max_sharpe()

If you print these weights, you will get quite an ugly result, because they will be the raw output from the optimizer. As such, it is recommended that you use the `clean_weights()` method, which truncates tiny weights to zero and rounds the rest:

In [76]:
cleaned_weights = ef.clean_weights()
ef.save_weights_to_file("efficient_frontier_weights.json")  # saves to file
print(cleaned_weights)

OrderedDict([('Bank Rakyat Indonesia', 0.0), ('Bank Mandiri', 1.0), ('Telkom Indonesia', 0.0)])


In [77]:
cleaned_weights

OrderedDict([('Bank Rakyat Indonesia', 0.0),
             ('Bank Mandiri', 1.0),
             ('Telkom Indonesia', 0.0)])

If we want to know the expected performance of the portfolio with optimal weights `w`, we can use the `portfolio_performance()` method:

In [78]:
ef.portfolio_performance(verbose=True)

Expected annual return: 6.2%
Annual volatility: 30.9%
Sharpe Ratio: 0.20


(np.float64(0.06190705429097587),
 np.float64(0.30927512917439853),
 np.float64(0.2001682270935754))

## Dealing with many negligible weights
From experience, I have found that mean-variance optimization often sets many of the asset weights to be zero. This may not be ideal if you need to have a certain number of positions in your portfolio, for diversification purposes or otherwise.

To combat this, I have introduced an objective function which borrows the idea of regularisation from machine learning. Essentially, by adding an additional cost function to the objective, you can ‘encourage’ the optimizer to choose different weights (mathematical details are provided in the More on L2 Regularisation section). To use this feature, change the gamma parameter:

In [79]:
from pypfopt import objective_functions

ef = EfficientFrontier(mu, S)
ef.add_objective(objective_functions.L2_reg, gamma=0.1)
w = ef.max_sharpe()
ef.clean_weights()

/opt/anaconda3/envs/stock_trading/lib/python3.11/site-packages/pypfopt/efficient_frontier/efficient_frontier.py:259: UserWarning: max_sharpe transforms the optimization problem so additional objectives may not work as expected.
  warnings.warn(


OrderedDict([('Bank Rakyat Indonesia', 0.0),
             ('Bank Mandiri', 1.0),
             ('Telkom Indonesia', 0.0)])

In [80]:
ef.portfolio_performance(verbose=True)

Expected annual return: 6.2%
Annual volatility: 30.9%
Sharpe Ratio: 0.20


(np.float64(0.06190705429097587),
 np.float64(0.30927512917439853),
 np.float64(0.2001682270935754))

# Post-processing weights
In practice, we then need to convert these weights into an actual allocation, telling you how many shares of each asset you should purchase. This is discussed further in Post-processing weights, but we provide an example below:

In [81]:
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

latest_prices = get_latest_prices(df)
da = DiscreteAllocation(w, latest_prices, total_portfolio_value=3_000_000)
allocation, leftover = da.lp_portfolio()

print(allocation)

{'Bank Mandiri': 638}


In [82]:
import pandas as pd

def allocation_to_df(allocation, latest_prices):
    # Create lists to store results
    stocks = []
    shares = []
    last_prices = []
    costs = []
    
    for stock, num_shares in allocation.items():
        price = latest_prices[stock]
        cost = num_shares * price
        stocks.append(stock)
        shares.append(num_shares)
        last_prices.append(price)
        costs.append(cost)
        
    df = pd.DataFrame({
        'Stock': stocks,
        'Shares': shares,
        'Last Price (IDR)': last_prices,
        'Cost(IDR)': costs
    })
    return df

# Example usage:
allocation_df = allocation_to_df(allocation, latest_prices)
allocation_df


,Stock,Shares,Last Price (IDR),Cost(IDR)
0,Bank Mandiri,638,4700.0,2998600.0
